# AI-READI Environmental and Clinical EDA (Full Cohort N=2,231)

This notebook explores environmental variables (temperature, humidity, light, PM2.5, VOC, NOx) from the Lee Lab Anura sensors across the **full cohort of N=2,231 participants** and correlates them with clinical comorbidities, age groups, and diabetes status.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from io import BytesIO
from azure.storage.blob import BlobServiceClient
from scipy import stats

import sys
sys.path.append(os.path.abspath('..'))
from config import azure_config

# Initialize Azure Blob Service Client
blob_service_client = BlobServiceClient.from_connection_string(azure_config.CONNECTION_STRING)
container_client = blob_service_client.get_container_client(azure_config.CONTAINER_NAME)


## 1. Load Full Cohort Participants and Stratify

In [ ]:
# Load master participants from Azure Blob Storage
participants_blob = f'{azure_config.STUDY_ID}/dataset/participants.tsv'
bc = container_client.get_blob_client(participants_blob)
df_participants = pd.read_csv(BytesIO(bc.download_blob().readall()), sep='	')

group_mapping = {
    'healthy': 'healthy',
    'pre_diabetes_lifestyle_controlled': 'no diabetes',
    'oral_medication_and_or_non_insulin_injectable_medication_controlled': 'controlled diabetes',
    'insulin_dependent': 'insulin dependent',
    'insulin_controlled': 'insulin dependent'
}
df_participants['diabetes_status'] = df_participants['study_group'].map(group_mapping)

def stratify_age(age):
    if age < 55:
        return '40-54'
    elif age < 70:
        return '55-69'
    else:
        return '70+'

df_participants['age_group'] = df_participants['age'].apply(stratify_age)
print(f"Loaded {len(df_participants)} participants.")
print(df_participants[['person_id', 'age', 'age_group', 'study_group', 'diabetes_status']].head())


## 2. Load Full Environmental Dataset (N=2,231)

In [ ]:
# Load full processed environmental dataset (N=2,231)
try:
    df_env_summary = pd.read_csv('../data/samples/full_env_processed.csv')
    print(f"Loaded full environmental dataset with {len(df_env_summary)} participants.")
except FileNotFoundError:
    print("Preprocessed full env dataset not found. Generating fallback...")

df_merged = pd.merge(df_participants, df_env_summary, on='person_id', how='inner')
print(f"Merged cohort has {len(df_merged)} participants.")
df_merged.head()


## 3. Overall Environmental Distribution Summary (N=2,231)

In [ ]:
env_cols = ['mean_temp', 'mean_hum', 'mean_light', 'mean_pm25', 'mean_voc', 'mean_nox']
print(df_merged[env_cols].describe().round(3))

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
titles = ['Temperature (°C)', 'Humidity (%)', 'Light (lch0)', 'PM2.5 (µg/m³)', 'VOC Index (ppb)', 'NOx Index (ppb)']

for ax, col, title in zip(axes.flat, env_cols, titles):
    sns.histplot(df_merged[col].dropna(), kde=True, ax=ax, color='skyblue', bins=25)
    ax.set_title(title)
    ax.axvline(df_merged[col].median(), color='red', linestyle='--', label=f'Median: {df_merged[col].median():.2f}')
    ax.legend()

plt.tight_layout()
plt.show()


## 4. Correlate Environmental Variables with Diabetes Status

In [ ]:
diab_order = ['healthy', 'no diabetes', 'controlled diabetes', 'insulin dependent']
print("--- MEAN METRICS BY DIABETES STATUS ---")
print(df_merged.groupby('diabetes_status')[env_cols].mean().round(3))

print("\n--- KRUSKAL-WALLIS SIGNIFICANCE TESTS ---")
for col in env_cols:
    groups = [df_merged[df_merged['diabetes_status']==g][col].dropna().values for g in diab_order]
    stat, p = stats.kruskal(*groups)
    print(f"{col:12s}: H={stat:6.3f}, p-value={p:.4e}")

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, col, title in zip(axes.flat, env_cols, titles):
    sns.boxplot(data=df_merged, x='diabetes_status', y=col, order=diab_order, ax=ax, palette='Set2')
    ax.set_title(f"{title} by Diabetes Status")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right')

plt.tight_layout()
plt.show()


## 5. Correlate Environmental Variables with Age Groups

In [ ]:
age_order = ['40-54', '55-69', '70+']
print("--- MEAN METRICS BY AGE GROUP ---")
print(df_merged.groupby('age_group')[env_cols].mean().round(3))

print("\n--- KRUSKAL-WALLIS SIGNIFICANCE TESTS ---")
for col in env_cols:
    groups = [df_merged[df_merged['age_group']==g][col].dropna().values for g in age_order]
    stat, p = stats.kruskal(*groups)
    print(f"{col:12s}: H={stat:6.3f}, p-value={p:.4e}")

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, col, title in zip(axes.flat, env_cols, titles):
    sns.boxplot(data=df_merged, x='age_group', y=col, order=age_order, ax=ax, palette='Set3')
    ax.set_title(f"{title} by Age Group")

plt.tight_layout()
plt.show()


## 6. Analysis across All 12 Combinations (Diabetes Status × Age Group)

In [ ]:
combo_matrix = df_merged.groupby(['diabetes_status', 'age_group'])[env_cols].mean().round(3)
print("=== FULL POPULATION MATRIX ACROSS ALL 12 COMBINATIONS (N=2,231) ===")
print(combo_matrix)

# Heatmap of PM2.5 across all 12 combinations
pm25_pivot = df_merged.pivot_table(index='diabetes_status', columns='age_group', values='mean_pm25', aggfunc='mean')
pm25_pivot = pm25_pivot.reindex(index=diab_order, columns=age_order)

plt.figure(figsize=(8, 6))
sns.heatmap(pm25_pivot, annot=True, fmt='.2f', cmap='YlOrRd')
plt.title('Mean PM2.5 (µg/m³) across All 12 Combinations (Full Cohort N=2,231)')
plt.show()


## 7. Clinical Data (Other Diseases & Comorbidities)

In [ ]:
conditions_blob_path = f"{azure_config.STUDY_ID}/dataset/clinical_data/condition_occurrence.csv"
blob_client = container_client.get_blob_client(conditions_blob_path)

print("Downloading condition_occurrence.csv...")
download_stream = blob_client.download_blob()
df_conditions = pd.read_csv(BytesIO(download_stream.readall()), usecols=['person_id', 'condition_concept_id', 'condition_source_value'])
print(f"Loaded {len(df_conditions)} condition occurrences across full cohort.")

conditions_per_person = df_conditions.groupby('person_id')['condition_source_value'].nunique().reset_index()
conditions_per_person.columns = ['person_id', 'num_comorbidities']

df_merged_clin = pd.merge(df_merged, conditions_per_person, on='person_id', how='left')
df_merged_clin['num_comorbidities'] = df_merged_clin['num_comorbidities'].fillna(0)

plt.figure(figsize=(10, 6))
sns.boxplot(data=df_merged_clin, x='diabetes_status', y='num_comorbidities', hue='age_group', order=diab_order, hue_order=age_order)
plt.title('Number of Comorbidities by Diabetes Status and Age Group (Full N=2,280)')
plt.xticks(rotation=20)
plt.show()

combo_clin = df_merged_clin.groupby(['diabetes_status', 'age_group'])['num_comorbidities'].mean().round(2).unstack()
combo_clin = combo_clin.reindex(index=diab_order, columns=age_order)
print("=== MEAN COMORBIDITIES ACROSS ALL 12 COMBINATIONS ===")
print(combo_clin)
